# propygator — a five-minute orbit demo

`propygator` propagates a satellite state forward in time under a configurable
force model and gives you back a `Trajectory` you can inspect, plot, and export.

This is a quick tour of **four contrasting orbits**, each making one point:

1. **A normal LEO** — the everyday case and the standard outputs.
2. **A decaying orbit** — the propagator catches re-entry and stops gracefully.
3. **A hyperbolic escape** — unbound orbits terminate safely, not at infinity.
4. **A Molniya orbit** — a different regime plus the full spacecraft configuration.

Everything is SI (metres, m/s, seconds). Importing `propygator` does **not** start
the Java VM — the first propagation does, lazily.

In [ ]:
%matplotlib inline
from pathlib import Path

import numpy as np

import propygator as pgr

MU = 3.986004418e14  # Earth GM, m^3/s^2
R_EARTH = 6_378_137.0  # WGS84 equatorial radius, m

print("propygator", pgr.__version__)

## 1 · A normal LEO

A circular, ISS-like orbit at ~420 km and 51.6° inclination. A `State` is a
Cartesian position + velocity at an `Epoch`, in an explicit `Frame`.

In [ ]:
r = R_EARTH + 420_000.0
v = np.sqrt(MU / r)  # circular speed
inc = np.radians(51.6)

iss = pgr.State(
    pgr.Epoch.from_iso("2024-01-01T00:00:00"),
    np.array([r, 0.0, 0.0]),
    np.array([0.0, v * np.cos(inc), v * np.sin(inc)]),
    pgr.Frame.EME2000,
)

# No force_models given -> the leo_default preset (70x70 gravity, Sun/Moon third
# body, drag, SRP). One day, sampled every 60 s.
iss_traj = pgr.propagate_numerical(iss, duration=86400, output_step=60)

print(f"{len(iss_traj)} samples")
print("forces  :", iss_traj.metadata["force_models"])
print("stopped early?", iss_traj.metadata.get("terminated", False))  # ran the full day

In [ ]:
pgr.plot_summary(iss_traj);  # ground track + altitude + speed, one figure

In [ ]:
pgr.plot_3d(iss_traj).show()  # interactive 3-D — drag to rotate, scroll to zoom

The `metadata` is a full reproducibility record (versions, integrator, the exact
forces that acted). A normal run carries no `terminated` key — keep an eye on that
in the next two scenarios.

## 2 · A decaying orbit → graceful re-entry

Drop the same orbit to **140 km**. With drag on it spirals in within hours. Rather
than crash or return nonsense, the propagator **stops itself at re-entry and tells
you** — `duration` is just an upper bound.

In [ ]:
r = R_EARTH + 140_000.0
v = np.sqrt(MU / r)

decaying = pgr.State(
    pgr.Epoch.from_iso("2024-01-01T00:00:00"),
    np.array([r, 0.0, 0.0]),
    np.array([0.0, v * np.cos(inc), v * np.sin(inc)]),
    pgr.Frame.EME2000,
)

# You'll see a one-time warning that the orbit dropped below the drag model's
# free-molecular validity floor — that's the guard system being honest, not an error.
decay_traj = pgr.propagate_numerical(
    decaying, duration=129_600, output_step=600
)  # cap 1.5 d

m = decay_traj.metadata
print("terminated        :", m.get("terminated"))
print("termination_reason:", m.get("termination_reason"))
print("termination_epoch :", m.get("termination_epoch"))
print("samples           :", len(decay_traj), "(partial — up to re-entry)")

In [ ]:
pgr.plot_altitude(decay_traj);  # altitude spiraling down to re-entry

`terminated=True`, `termination_reason="reentry"`, and the trajectory holds the
samples up to the crossing. Every downstream verb (`plot_*`, `export_*`) works on a
partial trajectory unchanged.

## 3 · A hyperbolic escape → safely bounded

Give it **more than escape speed**: a genuinely unbound (e > 1) orbit. Orekit
propagates the hyperbola faithfully, and propygator's escape backstop stops it
cleanly at the Earth-Moon gravity-parity radius (~327,000 km) instead of running
out to absurd distances.

In [ ]:
hyperbolic = pgr.State(
    pgr.Epoch.from_iso("2024-01-01T00:00:00"),
    np.array([7.0e6, 0.0, 0.0]),
    np.array([0.0, 11_500.0, 0.0]),  # > escape speed at 7,000 km
    pgr.Frame.EME2000,
)
print(
    "eccentricity:",
    round(hyperbolic.to_keplerian().eccentricity, 3),
    "(> 1 -> unbound)",
)

esc_traj = pgr.propagate_numerical(hyperbolic, duration=10 * 86400, output_step=3600)

m = esc_traj.metadata
print("termination_reason:", m.get("termination_reason"))
print("termination_epoch :", m.get("termination_epoch"))
print("samples           :", len(esc_traj))

In [ ]:
pgr.plot_3d(esc_traj).show()  # the climb-out; Earth is the small sphere at center

`termination_reason="escape"`. The escape backstop is what makes propagating
unbound orbits safe — the run terminates at a physical boundary instead of integrating
forever.

## 4 · A Molniya orbit + a full spacecraft model

A very different regime: a **highly eccentric, 12-hour Molniya** orbit at the 63.4°
critical inclination. And a fully specified spacecraft — a **box bus with solar
panels**, a **density-varying drag table** (`VariableCd`), and an **attitude law**.
This shows the configuration depth in one shot.

In [ ]:
a = 26_554_000.0  # semi-major axis (12 h period)
r_p = R_EARTH + 600_000.0  # perigee radius
e = 1.0 - r_p / a
v_p = np.sqrt(MU * (2.0 / r_p - 1.0 / a))  # vis-viva speed at perigee
inc_m = np.radians(63.4)

molniya = pgr.State(
    pgr.Epoch.from_iso("2024-01-01T00:00:00"),
    np.array([r_p, 0.0, 0.0]),
    np.array([0.0, v_p * np.cos(inc_m), v_p * np.sin(inc_m)]),
    pgr.Frame.EME2000,
)
print(
    f"e = {e:.3f}, perigee {(r_p - R_EARTH) / 1e3:.0f} km, "
    f"apogee {(a * (1 + e) - R_EARTH) / 1e3:.0f} km"
)

mol_traj = pgr.propagate_numerical(
    molniya,
    duration=12 * 3600,  # ~one revolution
    output_step=120,
    spacecraft=pgr.SpacecraftConfig(
        mass_kg=1500,
        geometry=pgr.SpacecraftGeometry.box_and_panels(
            x_length_m=2.4,
            y_length_m=1.8,
            z_length_m=3.0,
            solar_array_area_m2=12.0,
            drag_coefficient=pgr.VariableCd.sphere_default(),
        ),
    ),
    attitude=pgr.InPlaneTracking(),
)
print(len(mol_traj), "samples")

In [ ]:
pgr.plot_ground_track(mol_traj);  # the distinctive high-eccentricity ground track

In [ ]:
pgr.plot_3d(mol_traj).show()  # the big tilted ellipse

Near apogee (well above the drag table's ceiling) you may see a soft one-time note
that drag is negligible there — the same guard system as scenario 2, just the quiet
end of it.

## Hand it off

`export_all` bundles any run into shareable files — a summary PNG, an interactive
3-D HTML, and a CSV (with a metadata header) — and returns where it wrote them.

In [ ]:
outputs = pgr.export_all(
    iss_traj,
    output_dir=Path("./demo_output"),
    frames_3d=[pgr.Frame.EME2000, pgr.Frame.ITRF],
)
outputs

---

That's the loop: **state → `propagate_numerical` → `Trajectory` → inspect / plot /
export**, across four very different orbits. For the step-by-step API walkthrough
see `02_numerical_propagation.ipynb`; the full contract (every preset, validation
rule, and output column) is in `docs/features.md` §1.1.